# Odor Classifier Training
5-class multi-label odor prediction: fruity, sweet, sulfurous, woody, fatty

In [1]:
%pip install scikit-learn

In [2]:
!git clone https://github.com/FufanLu/molecular-foundation-model.git

Cloning into 'molecular-foundation-model'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 64 (delta 17), reused 57 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 211.00 KiB | 19.18 MiB/s, done.
Resolving deltas: 100% (17/17), done.


### Upload embeddings

In [3]:
import os
os.makedirs('molecular-foundation-model/data/processed/embeddings', exist_ok=True)
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    !mv {name} molecular-foundation-model/data/processed/embeddings/

Saving leffingwell_embeddings.pkl to leffingwell_embeddings.pkl


### Load data + encode labels

In [5]:
%pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 74.8 MB/s eta 0:00:00


In [6]:
import sys
sys.path.append('molecular-foundation-model')

from src.evaluation.similarity import load_embeddings
from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

embeddings = load_embeddings()
df = load_leffingwell()
df = clean_smiles(df)
df = encode_labels(df)

print(f"{len(embeddings)} embeddings, {len(df)} molecules")
label_distribution(df)

3522 -> 3522 valid molecules (0 removed)
3519 embeddings, 3522 molecules
Class         Count
--------------------
fruity         2353
sweet          1497
sulfurous       921
woody           968
fatty           749
--------------------
Multi-label:  2281
Total:        3522


### Train Baseline MLP (on frozen embeddings)

In [7]:
from src.classifier.train import split_data, train_baseline, evaluate_baseline

compounds, X, Y, train_idx, test_idx = split_data(embeddings, df, test_size=0.2, seed=42)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

model, best_f1 = train_baseline(
    X[train_idx], Y[train_idx],
    X[test_idx], Y[test_idx],
    epochs=100, lr=1e-3, patience=15
)

Train: 2815, Test: 704
  epoch   0  loss 0.6186  f1_macro 0.2827  best 0.2827
  epoch  10  loss 0.4738  f1_macro 0.5760  best 0.6097
  epoch  20  loss 0.4486  f1_macro 0.6179  best 0.6225
  epoch  30  loss 0.4282  f1_macro 0.6334  best 0.6334
  epoch  40  loss 0.4103  f1_macro 0.6268  best 0.6395
  epoch  50  loss 0.3949  f1_macro 0.6331  best 0.6501
  epoch  60  loss 0.3762  f1_macro 0.6335  best 0.6501
  epoch  70  loss 0.3688  f1_macro 0.6346  best 0.6522
  early stop at epoch 76


In [8]:
evaluate_baseline(model, X[test_idx], Y[test_idx], ALL_5_CLASSES)


Class             F1
---------------------
fruity        0.8481
sweet         0.6385
sulfurous     0.6367
woody         0.5167
fatty         0.6212
---------------------
macro avg     0.6522


0.652245028249654

### Save baseline model

In [9]:
import torch
torch.save(model.state_dict(), 'odor_baseline.pt')
from google.colab import files
files.download('odor_baseline.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## LoRA Fine-tuning (optional, needs GPU)

In [10]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

GPU: NVIDIA A100-SXM4-40GB


In [11]:
%pip install peft

In [17]:
%pip install unimol_tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.9/120.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 38.6 MB/s eta 0:00:00


In [ ]:
import torch
from unimol_tools import UniMolRepr

clf = UniMolRepr(data_type='molecule', remove_hs=False)
model = clf.model

print("Attention-related Linear layers:")
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if any(s in name.lower() for s in ['query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj', 'attn']):
            print(f"  {name}")

2026-07-14 08:21:17 | unimol_tools/weights/weighthub.py | 54 | INFO | Uni-Mol Tools | Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
INFO:Uni-Mol Tools:Weights will be downloaded to default directory: /usr/local/lib/python3.12/dist-packages/unimol_tools/weights
2026-07-14 08:21:17 | unimol_tools/weights/weighthub.py | 71 | INFO | Uni-Mol Tools | Downloading mol_pre_all_h_220816.pt
INFO:Uni-Mol Tools:Downloading mol_pre_all_h_220816.pt
DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=3 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c408068bd40>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7c4061043250> server_hostname='huggingface.co' timeout=3
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7c4

DEBUG:filelock:Attempting to acquire lock 136615846249488 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:filelock:Lock 136615846249488 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to release lock 136615846249488 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Lock 136615846249488 released on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/.gitignore.lock
DEBUG:filelock:Attempting to acquire lock 136615846250016 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol_pre_all_h_220816.pt.lock
DEBUG:filelock:Lock 136615846250016 acquired on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol_pre_all_h_220816.pt.lock
DEBUG:filelock:Attempting to release lock 136615846250016 on /usr/local/lib/python3.12/dist-packages/unimol_tools/weights/.cache/huggingface/download/mol_pre_all_h_220816.pt.lock
DEBUG:filelock:Lock 13661584625

In [ ]:
from src.classifier.model import apply_lora, OdorClassifier

n = apply_lora(model, r=4, alpha=8, target_names=['q_proj', 'k_proj', 'v_proj'])
print(f"LoRA applied to {n} layers")

classifier = OdorClassifier(model, hidden_dim=128, num_classes=5, dropout=0.2)
classifier = classifier.cuda() if torch.cuda.is_available() else classifier

trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from unimol_hf import UnimolDataCollator
import numpy as np

collator = UnimolDataCollator(pad_token_id=tokenizer.pad_token_id)

train_smiles = [df.iloc[i]['smiles'] for i in train_idx]
test_smiles = [df.iloc[i]['smiles'] for i in test_idx]

print(f"Tokenizing {len(train_smiles)} train + {len(test_smiles)} test SMILES...")
train_tokens = [tokenizer.encode(s) for s in train_smiles]
test_tokens = [tokenizer.encode(s) for s in test_smiles]

print("Done.")

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
device = next(classifier.parameters()).device

epochs = 30
batch_size = 16
train_y = torch.tensor(Y[train_idx], dtype=torch.float32).to(device)
test_y = torch.tensor(Y[test_idx], dtype=torch.float32).to(device)

for epoch in range(epochs):
    classifier.train()
    perm = torch.randperm(len(train_tokens))
    total_loss = 0.0

    for i in range(0, len(train_tokens), batch_size):
        idx = perm[i:i+batch_size]
        batch_tokens = [train_tokens[j] for j in idx]
        batch_y = train_y[idx]

        batch = collator(batch_tokens)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        logits = classifier(batch)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0 or epoch == epochs - 1:
        classifier.eval()
        with torch.no_grad():
            test_batch = collator(test_tokens)
            test_batch = {k: v.to(device) for k, v in test_batch.items()}
            test_preds = torch.sigmoid(classifier(test_batch)).cpu().numpy()
            test_bin = (test_preds > 0.5).astype(int)
            f1 = f1_score(Y[test_idx], test_bin, average='macro', zero_division=0)
        print(f"epoch {epoch:3d}  loss {total_loss/max(1,(len(train_tokens)//batch_size)):.4f}  f1_macro {f1:.4f}")
        classifier.train()

In [ ]:
torch.save(classifier.state_dict(), 'odor_lora.pt')
files.download('odor_lora.pt')